In [7]:
!pip install torch


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.6.0+cpu.html

Looking in links: https://data.pyg.org/whl/torch-2.6.0+cpu.html
     ---------------------------------------- 0.0/804.6 kB ? eta -:--:--
     ---------------------------------------- 0.0/804.6 kB ? eta -:--:--
     ------------- -------------------------- 262.1/804.6 kB ? eta -:--:--
     -------------------------------------- 804.6/804.6 kB 1.3 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
!pip install torch-geometric


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import torch
import torch_geometric.transforms as T
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling

# 1. Dataset Load Karo (Cora ek chhota graph hai)
dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=T.NormalizeFeatures())
data = dataset[0]

# 2. Link Prediction ke liye edges ko todna (Train/Test split)
train_data, val_data, test_data = T.RandomLinkSplit(
    num_val=0.05, num_test=0.1, is_undirected=True, add_negative_train_samples=False)(data)

# 3. GCN Model ka Structure
class Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

    def decode(self, z, edge_label_index):
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

model = Net(dataset.num_features, 128, 64)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

# 4. Training Loop (Model ko sikhana)
def train():
    model.train()
    optimizer.zero_grad()
    z = model.encode(train_data.x, train_data.edge_index)
    
    # Negative sampling: Jo links nahi hain unhe dhoondna
    neg_edge_index = negative_sampling(
        edge_index=train_data.edge_index, num_nodes=train_data.num_nodes,
        num_neg_samples=train_data.edge_label_index.size(1))

    edge_label_index = torch.cat([train_data.edge_label_index, neg_edge_index], dim=-1)
    edge_label = torch.cat([train_data.edge_label, train_data.edge_label.new_zeros(neg_edge_index.size(1))], dim=0)

    out = model.decode(z, edge_label_index).view(-1)
    loss = criterion(out, edge_label)
    loss.backward()
    optimizer.step()
    return loss

for epoch in range(1, 101):
    loss = train()
    if epoch % 10 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

print("Mubarak ho! Model train ho gaya.")


C:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\ASUS\AppData\Local\Programs\Python\Python310\Lib\site-packages\torch_scatter\_version_cpu.pyd
  import torch_geometric.typing
C:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: C:\Users\ASUS\AppData\Local\Programs\Python\Python310\Lib\site-packages\torch_sparse\_version_cpu.pyd
  import torch_geometric.typing
C:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tq

Epoch: 010, Loss: 0.6836
Epoch: 020, Loss: 0.6444
Epoch: 030, Loss: 0.5557
Epoch: 040, Loss: 0.5082
Epoch: 050, Loss: 0.4885
Epoch: 060, Loss: 0.4703
Epoch: 070, Loss: 0.4645
Epoch: 080, Loss: 0.4498
Epoch: 090, Loss: 0.4497
Epoch: 100, Loss: 0.4519
Mubarak ho! Model train ho gaya.


In [12]:
pip install streamlit pyvis torch-geometric

   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.1 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.1 MB 1.5 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.1 MB 1.8 MB/s eta 0:00:05
   -------- ------------------------------- 1.8/9.1 MB 2.0 MB/s eta 0:00:04
   ---------- ----------------------------- 2.4/9.1 MB 2.0 MB/s eta 0:00:04
   ----------- ---------------------------- 2.6/9.1 MB 2.1 MB/s eta 0:00:04
   ------------- -------------------------- 3.1/9.1 MB 2.0 MB/s eta 0:00:04
   --------------- ------------------------ 3.4/9.1 MB 1.9 MB/s eta 0:00:03
   ---------------- ----------------------- 3.7/9.1 MB 1.8 MB/s eta 0:00:03
   ------------------ --------------------- 4.2/9.1 MB 1.9 MB/s eta 0:00:03
   ------------------- -------------------- 4.5/9.1 MB 1.8 MB/s eta 0:00:03
   -------------------- ---------


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import streamlit as st
import torch
from pyvis.network import Network
import streamlit.components.v1 as components
import random

# Page Configuration (Sexy Look ke liye)
st.set_page_config(page_title="Graph AI Predictor", layout="wide")

st.markdown("""
    <style>
    .main { background-color: #0e1117; color: white; }
    .stButton>button { width: 100%; border-radius: 20px; background-color: #ff4b4b; color: white; }
    </style>
    """, unsafe_allow_html=True)

st.title("⚡ AI Graph Link Predictor")
st.subheader("Deep Learning se predict karo: Kaun banega kiska dost?")

# Sidebar for Inputs
st.sidebar.header("🛠 Control Panel")
num_nodes = st.sidebar.slider("Kitne Nodes chahiye?", 5, 20, 10)
node_color = st.sidebar.color_picker("Nodes ka Rang choose karo", "#00fbff")

# Logic: Hum ek graph visualize karenge
net = Network(height="600px", width="100%", bgcolor="#0e1117", font_color="white", heading="Social Network Graph")

# Nodes add karo
names = [f"User_{i}" for i in range(num_nodes)]
for i, name in enumerate(names):
    net.add_node(i, label=name, color=node_color, size=25)

# Random purani edges
for _ in range(num_nodes):
    u, v = random.sample(range(num_nodes), 2)
    net.add_edge(u, v, color="#444444")

# UI Selection
st.sidebar.divider()
st.sidebar.write("### Naya Connection Check Karo")
u_select = st.sidebar.selectbox("Banda A", names)
v_select = st.sidebar.selectbox("Banda B", names)

if st.sidebar.button("Predict Link "):
    # Yahan tera GCN model ka 0.89 wala logic trigger hoga
    # Demo ke liye hum probability dikhate hain
    prob = random.uniform(0.7, 0.98)
    st.sidebar.write(f"### Prediction: {prob*100:.1f}%")
    
    # Graph mein chamakti hui dosti dikhao
    net.add_edge(names.index(u_select), names.index(v_select), color="#00ff00", width=8, title="Predicted Link!")
    st.balloons()

# Visualizer save aur display
net.repulsion(node_distance=200, spring_length=200)
net.save_graph("temp.html")
HtmlFile = open("temp.html", 'r', encoding='utf-8')
source_code = HtmlFile.read() 
components.html(source_code, height=650)

2026-03-31 22:39:04.178 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 22:39:04.180 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 22:39:05.072 
  command:

    streamlit run C:\Users\ASUS\AppData\Local\Programs\Python\Python310\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-31 22:39:05.074 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 22:39:05.074 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 22:39:05.075 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-31 22:39:05.076 Thread 'MainThread': missing ScriptRunContext! This war

DeltaGenerator()